# 🚀 Deepfake Detection Model Training
## Production Model for HiddenLayer App

**Goal:** Train Xception model to detect deepfakes with >90% accuracy

**Data Sources:**
- Kaggle deepfakes (archive.zip - you need to upload this)
- TPDNE GAN faces (auto-downloaded)

**Output:** `deepfake_net.tflite` - Ready to deploy!

---

### ⚙️ INSTRUCTIONS:
1. **Runtime → Change runtime type → GPU (T4)**
2. **Upload `archive.zip`** to this Colab (left sidebar, Files)
3. **Run all cells** (Runtime → Run all)
4. **Wait 2-3 hours**
5. **Download `deepfake_net.tflite`** when complete

In [ ]:
# Cell 1: GPU Verification
import tensorflow as tf
print('=' * 60)
print('GPU VERIFICATION')
print('=' * 60)
print(f'TensorFlow: {tf.__version__}')
gpus = tf.config.list_physical_devices('GPU')
print(f'GPUs Detected: {len(gpus)}')
if len(gpus) > 0:
    for gpu in gpus:
        print(f'  ✅ {gpu}')
    print('\n🚀 GPU READY FOR TRAINING!')
else:
    print('\n❌ NO GPU! Go to Runtime → Change runtime type → GPU')
    raise SystemExit('GPU Required')

In [ ]:
# Cell 2: Install Dependencies
!pip install -q opencv-python requests pillow
print('✅ Dependencies installed')

In [ ]:
# Cell 3: Setup
import os
import shutil
import zipfile
import cv2
import numpy as np
import requests
import time
from tensorflow import keras

# Config
TARGET_SIZE = (299, 299)
BATCH_SIZE = 32
FROZEN_EPOCHS = 10
FINETUNE_EPOCHS = 15

WORKSPACE = 'training_data'
RAW_DIR = f'{WORKSPACE}/raw'
PROCESSED_DIR = f'{WORKSPACE}/processed'

# Create workspace
if os.path.exists(WORKSPACE):
    shutil.rmtree(WORKSPACE)
os.makedirs(f'{RAW_DIR}/real', exist_ok=True)
os.makedirs(f'{RAW_DIR}/fake', exist_ok=True)
os.makedirs(f'{PROCESSED_DIR}/real', exist_ok=True)
os.makedirs(f'{PROCESSED_DIR}/fake', exist_ok=True)

print('✅ Workspace ready')

---
## 📦 DATA COLLECTION

**IMPORTANT:** Upload `archive.zip` to this Colab before running the next cell!

In [ ]:
# Cell 4: Extract Kaggle Dataset
print('\n' + '=' * 60)
print('EXTRACTING KAGGLE DATASET')
print('=' * 60)

if not os.path.exists('archive.zip'):
    print('❌ archive.zip NOT FOUND!')
    print('   Please upload it using the Files panel (left sidebar)')
    raise FileNotFoundError('archive.zip missing')

print('Extracting archive.zip...')
temp = f'{WORKSPACE}/temp_kaggle'
with zipfile.ZipFile('archive.zip', 'r') as z:
    z.extractall(temp)

# Find real and fake folders
real_src, fake_src = None, None
for root, dirs, files in os.walk(temp):
    for d in dirs:
        path = os.path.join(root, d)
        d_lower = d.lower()
        if 'real' in d_lower and 'fake' not in d_lower and len(os.listdir(path)) > 5:
            real_src = path
        if 'fake' in d_lower and 'real' not in d_lower and len(os.listdir(path)) > 5:
            fake_src = path

if not real_src or not fake_src:
    print('❌ Could not find real/fake folders in archive.zip')
    raise ValueError('Invalid archive structure')

# Copy images
def copy_images(src, dst, limit):
    files = [f for f in os.listdir(src) if f.lower().endswith(('.jpg', '.jpeg', '.png'))]
    count = 0
    for f in files[:limit]:
        shutil.copy(os.path.join(src, f), os.path.join(dst, f))
        count += 1
    return count

real_count = copy_images(real_src, f'{RAW_DIR}/real', 2500)
fake_count = copy_images(fake_src, f'{RAW_DIR}/fake', 1500)

print(f'✅ Kaggle: {real_count} real, {fake_count} fake')
shutil.rmtree(temp)

In [ ]:
# Cell 5: Download TPDNE GAN Faces
print('\n' + '=' * 60)
print('DOWNLOADING GAN FACES FROM TPDNE')
print('=' * 60)

url = 'https://thispersondoesnotexist.com/'
headers = {'User-Agent': 'Mozilla/5.0'}
target = 1500
success = 0

for i in range(target):
    try:
        r = requests.get(url, headers=headers, timeout=10)
        if r.status_code == 200:
            with open(f'{RAW_DIR}/fake/tpdne_{i:04d}.jpg', 'wb') as f:
                f.write(r.content)
            success += 1
            if success % 100 == 0:
                print(f'   {success}/{target} downloaded')
        time.sleep(0.3)
    except:
        continue

print(f'✅ TPDNE: {success} GAN faces downloaded')
print(f'\n📊 Total: {real_count} real, {fake_count + success} fake')

---
## ✂️ PREPROCESSING

In [ ]:
# Cell 6: Face Detection and Cropping
print('\n' + '=' * 60)
print('FACE DETECTION')
print('=' * 60)

cascade = cv2.CascadeClassifier(cv2.data.haarcascades + 'haarcascade_frontalface_default.xml')

for label in ['real', 'fake']:
    src = f'{RAW_DIR}/{label}'
    dst = f'{PROCESSED_DIR}/{label}'
    
    files = [f for f in os.listdir(src) if f.lower().endswith(('.jpg', '.jpeg', '.png'))]
    print(f'\n{label.upper()}: Processing {len(files)} images...')
    
    saved = 0
    for i, fname in enumerate(files):
        img = cv2.imread(os.path.join(src, fname))
        if img is None:
            continue
        
        gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
        faces = cascade.detectMultiScale(gray, 1.1, 4, minSize=(80, 80))
        
        if len(faces) > 0:
            x, y, w, h = max(faces, key=lambda r: r[2]*r[3])
            margin = int(w * 0.2)
            x, y = max(0, x-margin), max(0, y-margin)
            w = min(img.shape[1]-x, w+2*margin)
            h = min(img.shape[0]-y, h+2*margin)
            
            face = cv2.resize(img[y:y+h, x:x+w], TARGET_SIZE)
            cv2.imwrite(os.path.join(dst, fname), face)
            saved += 1
        
        if (i + 1) % 500 == 0:
            print(f'   Processed {i+1}/{len(files)}, Saved {saved}')
    
    print(f'   ✅ {label}: {saved} faces extracted')

In [ ]:
# Cell 7: Balance Dataset
print('\n' + '=' * 60)
print('BALANCING DATASET')
print('=' * 60)

real_files = os.listdir(f'{PROCESSED_DIR}/real')
fake_files = os.listdir(f'{PROCESSED_DIR}/fake')

target_per_class = min(len(real_files), len(fake_files), 3000)

if len(real_files) > target_per_class:
    for f in real_files[target_per_class:]:
        os.remove(f'{PROCESSED_DIR}/real/{f}')

if len(fake_files) > target_per_class:
    for f in fake_files[target_per_class:]:
        os.remove(f'{PROCESSED_DIR}/fake/{f}')

final_real = len(os.listdir(f'{PROCESSED_DIR}/real'))
final_fake = len(os.listdir(f'{PROCESSED_DIR}/fake'))

print(f'✅ Balanced: {final_real} real, {final_fake} fake')
print(f'   Total training samples: {final_real + final_fake}')

---
## 🏋️ TRAINING (This will take 2-3 hours)

In [ ]:
# Cell 8: Load Data
print('\n' + '=' * 60)
print('LOADING DATA')
print('=' * 60)

# Training data
train_ds = keras.utils.image_dataset_from_directory(
    PROCESSED_DIR,
    validation_split=0.2,
    subset='training',
    seed=123,
    image_size=TARGET_SIZE,
    batch_size=BATCH_SIZE,
    label_mode='categorical'
)

# Validation data
val_ds = keras.utils.image_dataset_from_directory(
    PROCESSED_DIR,
    validation_split=0.2,
    subset='validation',
    seed=123,
    image_size=TARGET_SIZE,
    batch_size=BATCH_SIZE,
    label_mode='categorical'
)

# Preprocessing
def preprocess(images, labels):
    return keras.applications.xception.preprocess_input(images), labels

train_ds = train_ds.map(preprocess)
val_ds = val_ds.map(preprocess)

# Prefetch for performance
train_ds = train_ds.prefetch(tf.data.AUTOTUNE)
val_ds = val_ds.prefetch(tf.data.AUTOTUNE)

print('✅ Data loaded and preprocessed')

In [ ]:
# Cell 9: Build Model
print('\n' + '=' * 60)
print('BUILDING MODEL')
print('=' * 60)

from keras import mixed_precision
from keras.applications import Xception
from keras.layers import Dense, GlobalAveragePooling2D, Dropout
from keras.models import Model
from keras.optimizers import Adam
from keras.callbacks import EarlyStopping, ModelCheckpoint, ReduceLROnPlateau

# Mixed precision for faster training on GPU
mixed_precision.set_global_policy('mixed_float16')

with tf.device('/GPU:0'):
    base = Xception(weights='imagenet', include_top=False, input_shape=(299, 299, 3))
    base.trainable = False  # Freeze for phase 1
    
    x = base.output
    x = GlobalAveragePooling2D()(x)
    x = Dense(1024, activation='relu', dtype='float32')(x)
    x = Dropout(0.5)(x)
    out = Dense(2, activation='softmax', dtype='float32', name='predictions')(x)
    
    model = Model(base.input, out)

print(f'✅ Model built')
print(f'   Total params: {model.count_params():,}')
print(f'   Trainable: {sum([tf.size(w).numpy() for w in model.trainable_weights]):,}')

In [ ]:
# Cell 10: Phase 1 Training (Frozen Base)
print('\n' + '=' * 60)
print(f'PHASE 1: FROZEN BASE ({FROZEN_EPOCHS} epochs)')
print('=' * 60)

model.compile(
    optimizer=Adam(1e-4),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

history1 = model.fit(
    train_ds,
    epochs=FROZEN_EPOCHS,
    validation_data=val_ds,
    callbacks=[
        EarlyStopping('val_loss', patience=3, restore_best_weights=True),
        ReduceLROnPlateau('val_loss', factor=0.5, patience=2, verbose=1)
    ],
    verbose=1
)

phase1_acc = max(history1.history['val_accuracy'])
print(f'\n✅ Phase 1 Complete: Val Accuracy = {phase1_acc:.3f}')

In [ ]:
# Cell 11: Phase 2 Training (Fine-tuning)
print('\n' + '=' * 60)
print(f'PHASE 2: FINE-TUNING ({FINETUNE_EPOCHS} epochs)')
print('=' * 60)

# Unfreeze last 30 layers
base.trainable = True
for layer in base.layers[:-30]:
    layer.trainable = False

trainable = sum([tf.size(w).numpy() for w in model.trainable_weights])
print(f'Trainable params: {trainable:,}')

model.compile(
    optimizer=Adam(1e-5),  # Lower learning rate
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

history2 = model.fit(
    train_ds,
    epochs=FINETUNE_EPOCHS,
    validation_data=val_ds,
    callbacks=[
        EarlyStopping('val_loss', patience=2, restore_best_weights=True),
        ReduceLROnPlateau('val_loss', factor=0.3, patience=2, verbose=1)
    ],
    verbose=1
)

final_acc = max(history2.history['val_accuracy'])
print(f'\n✅ Phase 2 Complete: Val Accuracy = {final_acc:.3f}')

if final_acc >= 0.90:
    print('🎉 SUCCESS! Accuracy >= 90%')
elif final_acc >= 0.85:
    print('✅ Good! Accuracy >= 85%')
else:
    print('⚠️ Warning: Accuracy < 85%. Consider retraining with more data.')

---
## 📦 EXPORT TO TFLITE

In [ ]:
# Cell 12: Export TFLite
print('\n' + '=' * 60)
print('EXPORTING TO TFLITE')
print('=' * 60)

converter = tf.lite.TFLiteConverter.from_keras_model(model)
converter.optimizations = [tf.lite.Optimize.DEFAULT]
converter.target_spec.supported_types = [tf.float16]

tflite_model = converter.convert()

with open('deepfake_net.tflite', 'wb') as f:
    f.write(tflite_model)

size_mb = len(tflite_model) / 1024 / 1024
print(f'✅ Model exported: deepfake_net.tflite ({size_mb:.1f} MB)')
print(f'   FP16 quantized for mobile deployment')

In [ ]:
# Cell 13: Test TFLite Model
print('\n' + '=' * 60)
print('TESTING TFLITE MODEL')
print('=' * 60)

# Load TFLite model
interpreter = tf.lite.Interpreter(model_path='deepfake_net.tflite')
interpreter.allocate_tensors()

input_details = interpreter.get_input_details()
output_details = interpreter.get_output_details()

print(f'Input shape: {input_details[0]["shape"]}')
print(f'Output shape: {output_details[0]["shape"]}')

# Test with a random image
test_img = np.random.rand(1, 299, 299, 3).astype(np.float32)
test_img = (test_img * 255 - 127.5) / 127.5  # Normalize

interpreter.set_tensor(input_details[0]['index'], test_img)
interpreter.invoke()
output = interpreter.get_tensor(output_details[0]['index'])

print(f'\n✅ TFLite model working!')
print(f'   Sample output: {output[0]}')
print(f'   Real prob: {output[0][0]:.3f}, Fake prob: {output[0][1]:.3f}')

---
## 📥 DOWNLOAD THE MODEL

**`deepfake_net.tflite` is ready!**

1. Look in the Files panel (left sidebar)
2. Find `deepfake_net.tflite`
3. Right-click → Download
4. Copy it to: `HiddenLayer/app/src/main/assets/`
5. Rebuild your app: `./gradlew assembleDebug`
6. Install: `adb install -r app/build/outputs/apk/debug/app-debug.apk`

---

### 🎉 TRAINING COMPLETE!

**Final Stats:**

In [ ]:
# Cell 14: Summary
print('=' * 60)
print('TRAINING SUMMARY')
print('=' * 60)
print(f'\n📊 Dataset:')
print(f'   Real images: {final_real}')
print(f'   Fake images: {final_fake}')
print(f'   Total: {final_real + final_fake}')
print(f'\n🎯 Performance:')
print(f'   Phase 1 Accuracy: {phase1_acc:.1%}')
print(f'   Final Accuracy: {final_acc:.1%}')
print(f'\n📦 Model:')
print(f'   File: deepfake_net.tflite')
print(f'   Size: {size_mb:.1f} MB')
print(f'   Quantization: FP16')
print(f'\n✅ READY TO DEPLOY!')
print('   Download deepfake_net.tflite from the Files panel')
print('=' * 60)